In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

In [2]:
DIMENSION = '1K'

In [7]:
DATASET = f'../../datasets/paysim/{DIMENSION}/rawLog.csv'

In [ ]:
import csv

# Open the CSV file
with open(DATASET, mode='r', newline='') as infile:
    reader = csv.DictReader(infile)
    rows = list(reader)

In [4]:
chunk_acc = pd.read_csv(DATASET, chunksize=10000)
df = pd.concat(chunk_acc)

In [6]:
# Initialize balances dictionary
balances = {}

# Modify the rows
# for row in rows:
for idx, row in df.iterrows():
    try:
        orig = row['nameOrig']
        dest = row['nameDest']
        amount = float(row['amount'])
        action = row['action']

        # Use the last known balance or fallback to the CSV value
        old_balance_orig = balances.get(orig, float(row['oldBalanceOrig']))
        old_balance_dest = balances.get(dest, float(row['oldBalanceDest']))

        if action == 'CASH_IN':
            # dest gains money
            new_balance_dest = old_balance_dest - amount
            balances[dest] = new_balance_dest  # Update the live balance
            row['newBalanceDest'] = round(new_balance_dest, 2)
            row['oldBalanceDest'] = round(old_balance_dest, 2)  # also important to update oldBalanceDest!
        
        elif action == 'CASH_OUT' or action == 'DEBIT':
            # orig loses money
            new_balance_dest = old_balance_dest + amount
            balances[dest] = new_balance_dest
            row['newBalanceDest'] = round(new_balance_dest, 2)
            row['oldBalanceDest'] = round(old_balance_dest, 2)  #  also important to update oldBalanceOrig!            

    except ValueError:
        # Handle invalid values if needed
        row['newBalanceDest'] = ''
        row['newBalanceOrig'] = ''


In [9]:
# Initialize balances dictionary
balances = {}

# Modify the rows properly
for idx, row in df.iterrows():
    try:
        orig = row['nameOrig']
        dest = row['nameDest']
        amount = float(row['amount'])
        action = row['action']

        # Use the last known balance or fallback to the CSV value
        old_balance_orig = balances.get(orig, float(row['oldBalanceOrig']))
        old_balance_dest = balances.get(dest, float(row['oldBalanceDest']))

        if action == 'CASH_IN':
            # dest gains money
            new_balance_dest = old_balance_dest - amount
            balances[dest] = new_balance_dest

            df.at[idx, 'newBalanceDest'] = round(new_balance_dest, 2)
            df.at[idx, 'oldBalanceDest'] = round(old_balance_dest, 2)

        elif action == 'CASH_OUT' or action == 'DEBIT':
            # orig loses money
            new_balance_dest = old_balance_dest + amount
            balances[dest] = new_balance_dest

            df.at[idx, 'newBalanceDest'] = round(new_balance_dest, 2)
            df.at[idx, 'oldBalanceDest'] = round(old_balance_dest, 2)

    except ValueError:
        df.at[idx, 'newBalanceDest'] = ''
        df.at[idx, 'newBalanceOrig'] = ''


In [10]:
# Save to new CSV
NEW_DATASET = f"paysim{DIMENSION}.csv"
df.to_csv(NEW_DATASET, index=False)

In [6]:
import csv

# Write back to a new CSV file (or overwrite the original)
# NEW_DATASET = F"paysim{DIMENSION}.csv"

with open(DATASET, mode='w', newline='') as outfile:
    writer = csv.DictWriter(outfile, fieldnames=reader.fieldnames)
    writer.writeheader()
    for row in rows:
        writer.writerow(row)

FileNotFoundError: [Errno 2] No such file or directory: '../../datasets/paysim/1K/rawLog.csv'